DESAFIO 1

In [12]:
import boto3
from botocore.exceptions import ClientError

# Cliente S3 en us-west-2
s3 = boto3.client('s3', region_name='us-west-2')

In [13]:
def crear_bucket(nombre_bucket):
    try:
        s3.create_bucket(
            Bucket=nombre_bucket,
            CreateBucketConfiguration={
                'LocationConstraint': 'us-west-2'
            }
        )
        print(f" Bucket creado: {nombre_bucket}")
    except ClientError as e:
        print(f"Error: {e}")

In [14]:
bucket_name = "mi-bucket-angie-2026-01"
crear_bucket(bucket_name)

Error: An error occurred (ExpiredToken) when calling the CreateBucket operation: The provided token has expired.


In [4]:
with open("archivo.txt", "w") as f:
    f.write("Hola AWS")

In [5]:
def subir_archivo(ruta_local, bucket, nombre_s3):
    try:
        s3.upload_file(ruta_local, bucket, nombre_s3)
        print(f"Archivo subido: {nombre_s3}")
    except Exception as e:
        print(f"Error: {e}")

In [6]:
subir_archivo("archivo.txt", bucket_name, "archivo.txt")

Archivo subido: archivo.txt


In [7]:
def listar_archivos(bucket):
    try:
        response = s3.list_objects_v2(Bucket=bucket)
        
        if 'Contents' in response:
            print("Archivos en el bucket:")
            for obj in response['Contents']:
                print(f" - {obj['Key']}")
        else:
            print("Bucket vacío")
            
    except Exception as e:
        print(f"Error: {e}")

In [8]:
listar_archivos(bucket_name)

Archivos en el bucket:
 - archivo.txt


In [10]:
def descargar_archivo(bucket, nombre_s3, ruta_local):
    try:
        s3.download_file(bucket, nombre_s3, ruta_local)
        print(f" Descargado en: {ruta_local}")
    except Exception as e:
        print(f"Error: {e}")

In [11]:
descargar_archivo(bucket_name, "archivo.txt", "archivo_descargado.txt")

 Descargado en: archivo_descargado.txt


In [12]:
response = s3.list_buckets()

for bucket in response['Buckets']:
    print(bucket['Name'])

mi-bucket-angie-2026-01


DESAFIO 2

In [14]:
import os
from datetime import datetime

In [15]:
os.makedirs("mi_carpeta", exist_ok=True)

with open("mi_carpeta/archivo1.txt", "w") as f:
    f.write("Archivo 1")

with open("mi_carpeta/archivo2.txt", "w") as f:
    f.write("Archivo 2")

In [16]:
def backup_carpeta(ruta_carpeta, bucket):
    try:
        fecha = datetime.now().strftime("%Y-%m-%d")
        
        for archivo in os.listdir(ruta_carpeta):
            ruta_local = os.path.join(ruta_carpeta, archivo)
            
            if os.path.isfile(ruta_local):
                ruta_s3 = f"backup/{fecha}/{archivo}"
                
                s3.upload_file(ruta_local, bucket, ruta_s3)
                print(f"Subido: {ruta_s3}")
                
    except Exception as e:
        print(f"Error: {e}")

In [17]:
backup_carpeta("mi_carpeta", bucket_name)

Subido: backup/2026-04-10/archivo1.txt
Subido: backup/2026-04-10/archivo2.txt


DESAFIO 3

In [1]:
import boto3

sts = boto3.client("sts")
print(sts.get_caller_identity())

{'UserId': 'AROAYLLWMJD5AOB2V26HD:user4803418=Anyelina_Irene_Vilchis_Sierra', 'Account': '574163273978', 'Arn': 'arn:aws:sts::574163273978:assumed-role/voclabs/user4803418=Anyelina_Irene_Vilchis_Sierra', 'ResponseMetadata': {'RequestId': '752c3c0a-6ded-42d9-96ea-9e724aca9072', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '752c3c0a-6ded-42d9-96ea-9e724aca9072', 'x-amz-sts-extended-request-id': 'MTp1cy13ZXN0LTI6UzoxNzc1OTU5NjY0NTMxOlI6UzdHZVB6VzM=', 'content-type': 'text/xml', 'content-length': '496', 'date': 'Sun, 12 Apr 2026 02:07:44 GMT'}, 'RetryAttempts': 0}}


In [2]:
ec2 = boto3.client('ec2', region_name='us-west-2')

In [3]:
def listar_instancias():
    try:
        response = ec2.describe_instances()
        
        for reserva in response['Reservations']:
            for instancia in reserva['Instances']:
                
                instance_id = instancia['InstanceId']
                estado = instancia['State']['Name']
                tipo = instancia['InstanceType']
                ip_publica = instancia.get('PublicIpAddress', 'Sin IP')

                print(f"ID: {instance_id}")
                print(f"Estado: {estado}")
                print(f"Tipo: {tipo}")
                print(f"IP pública: {ip_publica}")
                print("-" * 40)

    except Exception as e:
        print(f"Error: {e}")

In [5]:
listar_instancias()

ID: i-0eb56b18c847b83a5
Estado: running
Tipo: t3.micro
IP pública: 54.218.255.182
----------------------------------------


DESAFIO 4

In [14]:
from datetime import datetime, timezone, timedelta

In [13]:
import boto3

s3 = boto3.client('s3', region_name='us-west-2')

In [15]:
def limpiar_bucket(bucket, dias=30):
    try:
        response = s3.list_objects_v2(Bucket=bucket)
        
        if 'Contents' not in response:
            print("Bucket vacío")
            return
        
        ahora = datetime.now(timezone.utc)
        
        for obj in response['Contents']:
            nombre = obj['Key']
            fecha_mod = obj['LastModified']
            
            diferencia = ahora - fecha_mod
            
            if diferencia > timedelta(days=dias):
                s3.delete_object(Bucket=bucket, Key=nombre)
                print(f"Eliminado: {nombre}")
            else:
                print(f"Se conserva: {nombre}")
                
    except Exception as e:
        print(f"Error: {e}")

In [16]:
limpiar_bucket("prueba-limpieza-angie-2026", 0)

Eliminado: archivo.txt
Eliminado: archivo1.txt


DESAFIO 5

In [17]:
cloudwatch = boto3.client('cloudwatch', region_name='us-west-2')

In [18]:
def enviar_metrica(nombre_metrica, valor):
    try:
        response = cloudwatch.put_metric_data(
            Namespace='MiApp',
            MetricData=[
                {
                    'MetricName': nombre_metrica,
                    'Value': valor,
                    'Unit': 'Count'
                }
            ]
        )
        print("Métrica enviada correctamente")
    except Exception as e:
        print(f"Error: {e}")

In [19]:
enviar_metrica("ArchivosSubidos", 5)

Métrica enviada correctamente


DESAFIO 6

In [20]:
iam = boto3.client('iam')

In [21]:
def crear_usuario(nombre_usuario):
    try:
        iam.create_user(UserName=nombre_usuario)
        print(f"Usuario creado: {nombre_usuario}")
    except Exception as e:
        print(f"Error: {e}")

In [22]:
def asignar_politica(nombre_usuario):
    try:
        iam.attach_user_policy(
            UserName=nombre_usuario,
            PolicyArn='arn:aws:iam::aws:policy/ReadOnlyAccess'
        )
        print(f"Política asignada a: {nombre_usuario}")
    except Exception as e:
        print(f"Error: {e}")

In [23]:
def crear_access_keys(nombre_usuario):
    try:
        response = iam.create_access_key(UserName=nombre_usuario)
        
        access_key = response['AccessKey']['AccessKeyId']
        secret_key = response['AccessKey']['SecretAccessKey']
        
        print("Access Key:", access_key)
        print("Secret Key:", secret_key)
        
    except Exception as e:
        print(f"Error: {e}")

In [24]:
usuario = "usuario-prueba-angie"

crear_usuario(usuario)
asignar_politica(usuario)
crear_access_keys(usuario)

Error: An error occurred (AccessDenied) when calling the CreateUser operation: User: arn:aws:sts::574163273978:assumed-role/voclabs/user4803418=Anyelina_Irene_Vilchis_Sierra is not authorized to perform: iam:CreateUser on resource: arn:aws:iam::574163273978:user/usuario-prueba-angie because no identity-based policy allows the iam:CreateUser action
Error: An error occurred (AccessDenied) when calling the AttachUserPolicy operation: User: arn:aws:sts::574163273978:assumed-role/voclabs/user4803418=Anyelina_Irene_Vilchis_Sierra is not authorized to perform: iam:AttachUserPolicy on resource: user usuario-prueba-angie because no identity-based policy allows the iam:AttachUserPolicy action
Error: An error occurred (AccessDenied) when calling the CreateAccessKey operation: User: arn:aws:sts::574163273978:assumed-role/voclabs/user4803418=Anyelina_Irene_Vilchis_Sierra is not authorized to perform: iam:CreateAccessKey on resource: user usuario-prueba-angie because no identity-based policy allows 

DESAFIO 7

In [25]:
import boto3
import json

ec2 = boto3.client('ec2', region_name='us-west-2')
s3 = boto3.client('s3', region_name='us-west-2')
lambda_client = boto3.client('lambda', region_name='us-west-2')

In [26]:
def obtener_ec2():
    instancias = []
    
    response = ec2.describe_instances()
    
    for reserva in response['Reservations']:
        for instancia in reserva['Instances']:
            instancias.append({
                "InstanceId": instancia['InstanceId'],
                "Tipo": instancia['InstanceType'],
                "Estado": instancia['State']['Name']
            })
    
    return instancias

In [27]:
def obtener_s3():
    buckets = []
    
    response = s3.list_buckets()
    
    for bucket in response['Buckets']:
        buckets.append({
            "Nombre": bucket['Name']
        })
    
    return buckets

In [28]:
def obtener_lambda():
    funciones = []
    
    response = lambda_client.list_functions()
    
    for func in response['Functions']:
        funciones.append({
            "Nombre": func['FunctionName'],
            "Runtime": func['Runtime']
        })
    
    return funciones

In [29]:
def generar_inventario():
    inventario = {
        "EC2": obtener_ec2(),
        "S3": obtener_s3(),
        "Lambda": obtener_lambda()
    }
    
    with open("aws_inventory.json", "w") as f:
        json.dump(inventario, f, indent=4)
    
    print("Inventario generado: aws_inventory.json")

In [30]:
generar_inventario()

Inventario generado: aws_inventory.json


DESAFIO 8

In [4]:
import boto3

s3 = boto3.client('s3', region_name='us-west-2')

In [1]:
with open("data.txt", "w") as f:
    f.write("linea 1\nlinea 2\nlinea 3\n")

In [2]:
def pipeline_s3(ruta_local, bucket_entrada, bucket_salida):
    try:
        nombre_archivo = ruta_local.split("/")[-1]
        
        # 1. Subir archivo
        s3.upload_file(ruta_local, bucket_entrada, nombre_archivo)
        print("Archivo subido")

        # 2. Leer desde S3
        response = s3.get_object(Bucket=bucket_entrada, Key=nombre_archivo)
        contenido = response['Body'].read().decode('utf-8')

        # 3. Procesar (contar líneas)
        num_lineas = len(contenido.splitlines())

        resultado = f"El archivo tiene {num_lineas} líneas"

        # 4. Guardar resultado en otro bucket
        nombre_salida = f"resultado_{nombre_archivo}"
        
        s3.put_object(
            Bucket=bucket_salida,
            Key=nombre_salida,
            Body=resultado
        )

        print("Resultado guardado en S3")

    except Exception as e:
        print(f"Error: {e}")

In [5]:
pipeline_s3("data.txt", "prueba-limpieza-angie-2026", "bucketsalida-aivs2207")

Archivo subido
Resultado guardado en S3


DESAFIO 9

In [6]:
ec2 = boto3.client('ec2', region_name='us-west-2')

In [7]:
def listar_volumenes():
    response = ec2.describe_volumes()
    
    volumenes = []
    
    for vol in response['Volumes']:
        volumenes.append(vol['VolumeId'])
    
    return volumenes

In [11]:
def crear_snapshots():
    try:
        volumenes = listar_volumenes()
        
        if not volumenes:
            print("No hay volumenes disponibles")
            return
        
        for vol_id in volumenes:
            response = ec2.create_snapshot(
                VolumeId=vol_id,
                Description=f"Snapshot automatico de {vol_id}"
            )
            
            print(f"Snapshot creado para {vol_id}: {response['SnapshotId']}")
    
    except Exception as e:
        print(f"Error: {e}")

In [12]:
crear_snapshots()

Snapshot creado para vol-09781854b29100e74: snap-02604b0f88b629cd3


DESAFIO 10

In [13]:
import boto3
import json

ec2 = boto3.client('ec2', region_name='us-west-2')
s3 = boto3.client('s3', region_name='us-west-2')

In [14]:
def obtener_reporte_ec2():
    reporte = []

    response = ec2.describe_instances()

    for reserva in response['Reservations']:
        for instancia in reserva['Instances']:
            reporte.append({
                "InstanceId": instancia['InstanceId'],
                "Tipo": instancia['InstanceType'],
                "Estado": instancia['State']['Name']
            })

    return reporte

In [15]:
def generar_reporte_json():
    data = {
        "EC2": obtener_reporte_ec2()
    }

    with open("reporte_ec2.json", "w") as f:
        json.dump(data, f, indent=4)

    print("Reporte generado")

In [16]:
def subir_reporte(bucket):
    s3.upload_file("reporte_ec2.json", bucket, "reporte_ec2.json")
    print("Reporte subido a S3")

In [18]:
generar_reporte_json()
subir_reporte("bucketsalida-aivs2207")

Reporte generado
Reporte subido a S3
